In [47]:
!pip install -q langchain==1.0.5 langchain-community==0.4.1 langchain-groq==1.0.0 langchain-google-genai==3.0.1 chromadb==1.3.4 pypdf python-dotenv pyngrok

In [48]:
!pip install langchain-core langchain-community langchain-google-genai langchain-text-splitters chromadb -q

In [49]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
print("API key loaded ✅")

API key loaded ✅


In [50]:
import json

with open("/content/bengaluru_places_rag.json") as f:
  data = json.load(f)

print(f"✅ Loaded {len(data['places'])} places from JSON")

✅ Loaded 37 places from JSON


In [51]:
import json
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings


places = data["places"]

documents = []
for place in places:
    content = f"""
Name: {place['name']}
Address: {place['address']}
Rating: {place['rating']} ({place['rating_count']} reviews)
Hours: {place['hours']}
Budget: {place['budget']['range']} — {place['budget']['notes']}
Description: {place['description']}
Highlights: {', '.join(place['highlights'])}
Best for (who): {', '.join(place['tags']['who'])}
Best for (time): {', '.join(place['tags']['time'])}
Best for (vibe): {', '.join(place['tags']['vibe'])}
""".strip()

    documents.append(Document(
    page_content=content,
    metadata={
        "name": place["name"],
        "budget_level": place["budget"]["level"],
        "who": ", ".join(place["tags"]["who"]),
        "time": ", ".join(place["tags"]["time"]),
        "vibe": ", ".join(place["tags"]["vibe"]),
        "rating": place["rating"],
        "address": place["address"],
        "latitude": place["latitude"],
        "longitude": place["longitude"]
    }
))
print(f"✅ {len(documents)} places loaded as documents")

splitter = CharacterTextSplitter(chunk_size=600, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f"✅ {len(chunks)} chunks created")

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GEMINI_API_KEY)

vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✅ ChromaDB vector store ready — Lila's knowledge base is live")

✅ 37 places loaded as documents
✅ 37 chunks created
✅ ChromaDB vector store ready — Lila's knowledge base is live


In [52]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

llm=ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0.3,
    max_tokens=5000
)

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are Lila, a warm and helpful travel and hangout companion for people in Bengaluru.
Based on the places information below, suggest the most relevant options.
For each place give the name, why it fits the mood, budget, and highlights.
Keep it friendly and concise.

Places information:
{context}

User's mood and request:
{question}

Your suggestions:
"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Lila's chain is ready")

✅ Lila's chain is ready


In [53]:
# Query 1
query = "I am going with friends, have a few hours, want a chill vibe, staying in Bengaluru"
response = qa_chain.invoke(query)
print("Query:", query)
print("Lila:", response)

# Query 2
query2 = "Solo trip, weekend, adventure mood, okay to go 100km outside Bengaluru"
response2 = qa_chain.invoke(query2)
print("\nQuery:", query2)
print("Lila:", response2)

# Query 3 — Priya's persona
query3 = "Chill spot tonight, not too expensive, near Indiranagar, group of 4"
response3 = qa_chain.invoke(query3)
print("\nQuery:", query3)
print("Lila:", response3)

Query: I am going with friends, have a few hours, want a chill vibe, staying in Bengaluru
Lila: **Elements Mall – Thanisandra**

- **Why it fits your vibe** – A calm, uncrowded spot perfect for a relaxed hang‑out with friends. No rush, just easy strolling, good food, and a chill cinema vibe.  
- **Budget** – ₹200–₹800 per person (free entry).  
- **Highlights** –  
  • Great food court with a variety of quick bites  
  • PVR cinema for a casual movie break  
  • Supermarket inside for any quick needs  
  • Quiet atmosphere, ideal for a few hours of laid‑back fun  

It’s the go‑to for a short, stress‑free outing in North Bengaluru, especially if you’re near Manyata Tech Park. Enjoy!

Query: Solo trip, weekend, adventure mood, okay to go 100km outside Bengaluru
Lila: **Rocksport Bengaluru – “Campfire & Rope‑Trek”**

- **Why it fits your mood**  
  A weekend escape that’s all about adventure and a touch of relaxation. You’ll get to try rope‑activities, rain‑dance, and a cozy campfire – pe

### Add Weather and Google Maps Integration

To fetch real-time weather, we'll use the OpenWeatherMap API. Please sign up for a free API key at [OpenWeatherMap](https://openweathermap.org/api) and add it to your Colab secrets as `OPENWEATHER_API_KEY`.

We'll also create a helper function to generate Google Maps links using the latitude and longitude data we already have.

In [66]:
import requests
from google.colab import userdata

OPENWEATHER_API_KEY = userdata.get('OPENWEATHER_API_KEY').strip()

print("OpenWeatherMap API key loaded (if available) ✅")

OpenWeatherMap API key loaded (if available) ✅


In [62]:
def get_weather(lat, lon, api_key):
    if not api_key:
        return "Weather API key not found. Cannot fetch weather."

    base_url = "http://api.openweathermap.org/data/2.5/weather"
    params = {
        "lat": lat,
        "lon": lon,
        "appid": api_key,
        "units": "metric" # For Celsius, use "imperial" for Fahrenheit
    }
    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status() # Raise an exception for HTTP errors
        weather_data = response.json()
        if weather_data and weather_data.get('main') and weather_data.get('weather'):
            temp = weather_data['main']['temp']
            description = weather_data['weather'][0]['description']
            return f"{temp}°C, {description.capitalize()}"
        else:
            return "Weather data not available."
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 401:
            return "Weather API key is unauthorized or invalid. Please check your OPENWEATHER_API_KEY."
        else:
            return f"Error fetching weather: {e}"
    except requests.exceptions.RequestException as e:
        return f"Error fetching weather: {e}"

In [67]:
import requests
from google.colab import userdata

EXCHANGERATE_API_KEY = userdata.get('EXCHANGERATE_API_KEY')

print("Exhangerate api  key loaded (if available) ✅")

Exhangerate api  key loaded (if available) ✅


In [68]:
import re

def get_exchange_rate(from_currency, to_currency, api_key):
    if not api_key:
        return "Currency API key not found. Cannot fetch exchange rates."

    # Using ExchangeRate-API as an example (replace with your chosen API endpoint)
    base_url = f"https://v6.exchangerate-api.com/v6/{api_key}/latest/{from_currency}"
    try:
        response = requests.get(base_url)
        response.raise_for_status() # Raise an exception for HTTP errors
        data = response.json()
        if data and data['result'] == 'success' and to_currency in data['conversion_rates']:
            return data['conversion_rates'][to_currency]
        else:
            print(f"Error: Exchange rate data not available for {to_currency}.")
            return None
    except requests.exceptions.RequestException as e:
        print(f"Error fetching exchange rate: {e}")
        return None

def convert_budget(budget_range_str, exchange_rate, to_currency='USD'):
    if not exchange_rate:
        return f"Budget: {budget_range_str} (conversion not available)"

    # Assuming budget_range_str is like '₹200–₹800 per person' or similar
    numbers = re.findall(r'(\d+(?:\.\d+)?)', budget_range_str.replace('₹', '').replace(',', ''))

    if numbers:
        converted_numbers = [float(n) * exchange_rate for n in numbers]
        # Format to 2 decimal places and prepend currency symbol
        currency_symbol = {'USD': '$', 'EUR': '€', 'GBP': '£'}.get(to_currency.upper(), f'{to_currency} ')
        if len(converted_numbers) == 2:
            return f"{currency_symbol}{converted_numbers[0]:.2f}–{currency_symbol}{converted_numbers[1]:.2f} per person"
        elif len(converted_numbers) == 1:
            return f"{currency_symbol}{converted_numbers[0]:.2f} per person"
    return f"Budget: {budget_range_str} (conversion failed)"

In [69]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# New enrichment function to handle weather, maps, and currency conversion
def enrich_documents_and_pass_question(input_data):
    docs = input_data["docs"]
    question = input_data["question"]
    target_currency = input_data["target_currency"]

    # Fetch exchange rate once for all documents
    exchange_rate = None
    # Check if CURRENCY_API_KEY is available and defined in global scope
    if target_currency and target_currency.upper() != "INR" and 'EXCHANGERATE_API_KEY' in globals() and globals()['EXCHANGERATE_API_KEY']:
        # Assuming base currency is INR for Bengaluru places
        exchange_rate = get_exchange_rate("INR", target_currency.upper(), globals()['EXCHANGERATE_API_KEY'])
        if not exchange_rate:
            print(f"Warning: Could not fetch exchange rate for {target_currency}. Budget conversion will not be applied.")

    enriched_docs_content = []
    for doc in docs:
        name = doc.metadata.get("name", "Unknown Place")
        latitude = doc.metadata.get("latitude")
        longitude = doc.metadata.get("longitude")

        additional_info_lines = []

        # Add weather (if OPENWEATHER_API_KEY is available and defined)
        if latitude is not None and longitude is not None:
            if 'OPENWEATHER_API_KEY' in globals() and globals()['OPENWEATHER_API_KEY']:
                weather_info = get_weather(latitude, longitude, globals()['OPENWEATHER_API_KEY'])
                additional_info_lines.append(f"Current Weather: {weather_info}")
            else:
                additional_info_lines.append("Current Weather: (OpenWeatherMap API key not set)")

        # Process the main content to convert budget
        original_content_lines = doc.page_content.split('\n')
        processed_content_lines = []
        budget_replaced = False

        for line in original_content_lines:
            if line.startswith('Budget: ') and not budget_replaced:
                budget_part_match = re.search(r'Budget: (.*?)(?: —|$)', line)
                budget_part = budget_part_match.group(1).strip() if budget_part_match else ""

                converted_budget_line = line # Default to original line
                if exchange_rate and target_currency.upper() != "INR" and budget_part:
                    converted_budget_text = convert_budget(budget_part, exchange_rate, target_currency.upper())
                    if '—' in line: # Append the notes if they exist
                         notes_part = line.split('—', 1)[1].strip()
                         converted_budget_line = f"{converted_budget_text} — {notes_part}"
                    else:
                         converted_budget_line = converted_budget_text
                    converted_budget_line = "Budget: " + converted_budget_line # Ensure "Budget: " prefix

                processed_content_lines.append(converted_budget_line)
                budget_replaced = True
            else:
                processed_content_lines.append(line)

        final_content = "\n".join(processed_content_lines)

        if additional_info_lines:
            final_content += f"\n\nExternal Info:\n{'-'*20}\n{'\n'.join(additional_info_lines)}\n{'-'*20}"

        enriched_docs_content.append(final_content)

    return {"context": "\n\n".join(enriched_docs_content), "question": question}

# Re-define Lila's chain to include both types of enrichment
qa_chain_with_all_enrichment = (
    RunnablePassthrough.assign(
        docs = (lambda x: x["question"]) | retriever, # Retrieve docs based on question
        question = lambda x: x["question"], # Pass the original question through
        target_currency = lambda x: x.get("target_currency", "INR") # Get target_currency from input or default to INR
    )
    | RunnableLambda(enrich_documents_and_pass_question) # This lambda takes the dict and returns dict{"context":..., "question":...}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Lila's chain with all enrichment (weather, maps, currency) is ready!")

✅ Lila's chain with all enrichment (weather, maps, currency) is ready!


In [70]:

query_input_1 = {
    "question": "I am going with friends, have a few hours, want a chill vibe, staying in Bengaluru. Also, tell me the current weather for these places.",
    "target_currency": "USD"
}
response_1 = qa_chain_with_all_enrichment.invoke(query_input_1)
print("Query:", query_input_1['question'])
print("Lila (Enriched with Currency & Weather):", response_1)


query_input_2 = {
    "question": "Solo trip, weekend, adventure mood, okay to go 100km outside Bengaluru. Please include the current weather."
}
response_2 = qa_chain_with_all_enrichment.invoke(query_input_2)
print("\nQuery:", query_input_2['question'])
print("Lila (Enriched, no specific currency):", response_2)


query_input_3 = {
    "question": "Chill spot tonight, not too expensive, near Indiranagar, group of 4. What's the current weather like at these locations?",
    "target_currency": "EUR"
}
response_3 = qa_chain_with_all_enrichment.invoke(query_input_3)
print("\nQuery:", query_input_3['question'])
print("Lila (Enriched with Currency & Weather):", response_3)

Query: I am going with friends, have a few hours, want a chill vibe, staying in Bengaluru. Also, tell me the current weather for these places.
Lila (Enriched with Currency & Weather): **Snow City Bengaluru**

| What it is | Why it fits your vibe | Budget | Highlights |
|------------|-----------------------|--------|------------|
| Indoor “snow” playground with slides, igloos, a glass‑bridge walk, and hot momos | Perfect for a chill, adventure‑filled hangout with friends – you’ll feel like you’re in a winter wonderland without the chill outside. | ₹600–₹700 per person (45‑min session) – a great deal for a few hours of fun. | • Snow slides & igloo<br>• Glass‑bridge walk<br>• Hot momos inside<br>• All gear (jacket, boots, gloves) provided |

**Why it’s a great pick for you**  
- **Mood**: Adventure + chill – you’ll laugh on the slides and then warm up with momos.  
- **Time**: A 45‑minute slot is plenty for a quick escape.  
- **Friends**: The space is big enough for a group, and the indo

In [65]:
print("Testing get_weather function directly for Bengaluru (example coordinates):")
bengaluru_lat = 12.9716
bengaluru_lon = 77.5946
test_weather_result = get_weather(bengaluru_lat, bengaluru_lon, OPENWEATHER_API_KEY)
print(test_weather_result)

# If you see an error here like 'Error fetching weather: 401 Client Error: Unauthorized',
# it means your OPENWEATHER_API_KEY is invalid or not correctly configured/activated on OpenWeatherMap.

Testing get_weather function directly for Bengaluru (example coordinates):
Weather API key is unauthorized or invalid. Please check your OPENWEATHER_API_KEY.
